In [1]:
from __future__ import annotations

import pickle
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any

import numpy as np
import os
import json
import gzip
import math

from stable_platform_matchings import Optimizer, InstanceGenerator
from stable_platform_matchings.optimization.options import OptimizerParams, SolverOptions
from stable_platform_matchings.domain.instance import Instance
from stable_platform_matchings.graphs.road_graphs import RoadGraph
from pprint import pprint

In [2]:
generator = InstanceGenerator(
    farmers_full_csv_path=Path("../data/farmers.csv"),
    farmers_14_csv_path=Path("../data/farmers_14.csv"),
    intermediaries_csv_path=Path("../data/intermediaries.csv"),
    graph_pkl_path=Path("../data/graph_0-14960_00_new.pickle"),
    alpha_json_path=Path("../data/precomputed_alpha.json"),
    sigmas_json_path=Path("../data/precomputed_sigmas.json")
)

In [7]:
generator.gen_intermediaries(n_intermediaries=40, seed=420)

generator.gen_calendar(seed=420, n_cycles=60, scale=1, cycle_length=14, buffer_cycles=1)

In [8]:
instance = generator.gen_instance(instance_id="67", day=800, n_hist_sets=8)

In [9]:
pprint(instance.to_snapshot())

{'constants': {'fruit_price_per_ton': 2513000,
               'lc_to_usd': 14500,
               'truck_capacity_tons': 9.0,
               'truck_cost_per_m': 2.625,
               'truck_fixed_cost': 800000},
 'distances': {'dirt': {'admiring_dirac': 139150.19647089497,
                        'admiring_dirac_d4_f4': 159730.39246441322,
                        'adoring_clarke': 119619.30486442352,
                        'affectionate_ride': 133363.5885802249,
                        'affectionate_ride_d2_f0': 169431.93006956417,
                        'affectionate_ride_d3_f0': 167909.33305835788,
                        'affectionate_ride_d4_f0': 133363.5885802249,
                        'affectionate_wing': 115993.8241254708,
                        'agitated_visvesvaraya': 72444.43924062155,
                        'agitated_visvesvaraya_d2_f1': 76244.96328080639,
                        'agitated_visvesvaraya_d3_f2': 70806.17152318508,
                        'awesome_shirley'

In [12]:
het_costs = {intermediary.id: instance.dist_to_mill[intermediary.id]*2 for intermediary in instance.intermediaries}
epsilons = {intermediary.id: 2.0 for intermediary in instance.intermediaries}

params = OptimizerParams(
    het_costs=het_costs,
    epsilons = epsilons,
    threads=14,
    vrp_time_limit_seconds=5
)

optimizer = Optimizer(instance, params)

Using new version of optimizer with pay_unmatched


============================= Optimizer Parameters =============================
---------------------------------- het_costs -----------------------------------
  {'admiring_dirac': 652792.5736130538,
   'adoring_clarke': 525545.9965054897,
   'affectionate_ride': 575407.0381205556,
   'affectionate_wing': 525801.9418127763,
   'agitated_visvesvaraya': 234657.9354308132,
   'awesome_shirley': 42642.82640631433,
   'beautiful_herschel': 472799.38541816745,
   'cool_davinci': 252665.8733820477,
   'crazy_banzai': 420776.28193562396,
   'dazzling_faraday': 623294.0211556281,
   'elegant_keller': 468740.8634407819,
   'exciting_roentgen': 356322.18326420913,
   'fervent_wu': 210002.32029161777,
   'focused_jennings': 283564.11222947954,
   'goofy_hoover': 364151.9691333831,
   'gracious_mclaren': 291285.25613163185,
   'heuristic_davinci': 652792.5736130538,
   'heuristic_ganguly': 293495.35095422855,
   'interesting_jones': 18715.117206

In [68]:
summary = optimizer.solve(options = SolverOptions(
    seed=67,
    strategy="heuristic_optimized"
))



================================ Solver Options ================================
  Seed                       67
  Strategy                   heuristic_optimized
  Structured Farmer Payments False
  Dominance Constraints      False
  Early Stop                 False
  Aggregate                  False
  Pay Unmatched              False
  Stabilize Branch Extrema   False


======================== Strategy: heuristic_optimized =========================
  Farmers                    27
  Intermediaries             14


============================== Branch Evaluation ===============================
  Forced matched:
    []
  Forced unmatched:
    []

----------------------------- Primal Solve Result ------------------------------
  Platform profit            12,366,121.222
  Max intermediary welfare   3,661,869.336
  Max farmer welfare         80,660,226.714
---------------------------- Lower-Bound Candidate -----------------------------
  Objective                  12,366,121.222
  Mini

RuntimeError: Primary primal solve (forced_lower_bound) produced no feasible solution; status=11.

In [ ]:
pprint(summary)

InstanceSummary(instance_snapshot={'constants': {'fruit_price_per_ton': 2513000,
                                                 'lc_to_usd': 14500,
                                                 'truck_capacity_tons': 9.0,
                                                 'truck_cost_per_m': 2.625,
                                                 'truck_fixed_cost': 800000},
                                   'distances': {'dirt': {'affectionate_ride': 24876.0504415688,
                                                          'affectionate_ride_d3_f1': 33215.910622535564,
                                                          'awesome_shirley': 21321.413203157168,
                                                          'awesome_shirley_d1_f0': 30692.053634606644,
                                                          'cool_davinci': 24876.0504415688,
                                                          'cool_davinci_d13_f0': 52740.630312331945,
                        

In [2]:
n_id = 1

sim_size = 1

def reset_quantities():
    return {farmer.id: farmer.quantity for farmer in Platform.farmers}

def reset_fixed_costs():
    return {intermediary.id: Platform.dist_to_mill[intermediary.id]*4 for intermediary in Platform.intermediaries}

In [3]:
results = []

instance_str = "2020-08-27"
Platform = Instance.from_yaml('../data/anon_14_day_instances/'+instance_str+'.yaml')
Platform = Instance.from_yaml('../data/anon_14_day_instances/'+instance_str+'.yaml', force_quantities=reset_quantities())

rng = np.random.default_rng(1)

epsilon = {int.id: 2.0 for int in Platform.intermediaries}

with open("../data/graph_0-14960_00_new.pickle", 'rb') as pickle_file:
    G = pickle.load(pickle_file)

Platform.set_graph(RoadGraph(G))
farmer_quantities = {farmer.id: farmer.quantity for farmer in Platform.farmers}
het_costs = reset_fixed_costs()
parameters = {
    "epsilon":epsilon, 
    "solver": "gurobi", 
    "het_costs": het_costs,
}

pprint(het_costs)
pprint(farmer_quantities)

farmer_dist_to_mill = {farmer.id: farmer.dist_to_mill for farmer in Platform.farmers}
farmer_dirt_to_mill = {farmer.id: farmer.dirt_to_mill for farmer in Platform.farmers}
farmer_paved_to_mill = {farmer.id: farmer.paved_to_mill for farmer in Platform.farmers}

pprint(farmer_dirt_to_mill)
pprint(farmer_paved_to_mill)

{'beautiful_bohr': 251156.24797256076,
 'competent_mayer': 406399.4018487107,
 'elated_haslett': 977691.1445767505,
 'elegant_gagarin': 627180.0496084697,
 'elegant_mendel': 348573.77294422314,
 'exciting_fermi': 493149.4427404745,
 'gallant_cerf': 647257.1157146845,
 'hopeful_sanderson': 823884.0054250445,
 'keen_visvesvaraya': 1194805.381073437,
 'laughing_mestorf': 840198.7682361763,
 'loving_engelbart': 1054403.388237939,
 'peaceful_austin': 868601.4376568777,
 'quizzical_elgamal': 870359.6940502283,
 'vigorous_mccarthy': 672948.548382829}
{'beautiful_bohr_14_12': 2.0,
 'competent_mayer_14_1': 2.0,
 'elegant_gagarin_16_1': 0.8,
 'elegant_gagarin_98_1': 1.9,
 'elegant_mendel_23_8': 2.0,
 'elegant_mendel_54_6': 3.6,
 'elegant_mendel_69_3': 3.2,
 'gallant_cerf_11_5': 2.3,
 'gallant_cerf_12_12': 1.2,
 'gallant_cerf_19_2': 1.0,
 'gallant_cerf_29_4': 3.5,
 'laughing_mestorf_101_1': 0.5,
 'laughing_mestorf_11_8': 1.7,
 'laughing_mestorf_24_19': 2.8,
 'laughing_mestorf_62_1': 0.5,
 'loving

In [4]:
params = OptimizerParams(
    het_costs=het_costs,
    epsilons=epsilon,
    threads=14
)
options = SolverOptions(
    aggregate=True,
    stabilize_branch_extrema=False
)

opt = Optimizer(Platform, params)

summary_vanilla = opt.solve(options)

results.append({
    "instance_str": instance_str,
    "cost": het_costs,
    "epsilon": epsilon,
    "farmer_quantities": farmer_quantities,
    "summary_vanilla": summary_vanilla,
    "farmer_dirt_to_mill": farmer_dirt_to_mill,
    "famer_paved_to_mill": farmer_paved_to_mill,
})




============================= Optimizer Parameters =============================
---------------------------------- het_costs -----------------------------------
  {'beautiful_bohr': 251156.24797256076,
   'competent_mayer': 406399.4018487107,
   'elated_haslett': 977691.1445767505,
   'elegant_gagarin': 627180.0496084697,
   'elegant_mendel': 348573.77294422314,
   'exciting_fermi': 493149.4427404745,
   'gallant_cerf': 647257.1157146845,
   'hopeful_sanderson': 823884.0054250445,
   'keen_visvesvaraya': 1194805.381073437,
   'laughing_mestorf': 840198.7682361763,
   'loving_engelbart': 1054403.388237939,
   'peaceful_austin': 868601.4376568777,
   'quizzical_elgamal': 870359.6940502283,
   'vigorous_mccarthy': 672948.548382829}
----------------------------------- epsilons -----------------------------------
  {'beautiful_bohr': 2.0,
   'competent_mayer': 2.0,
   'elated_haslett': 2.0,
   'elegant_gagarin': 2.0,
   'elegant_mendel': 2.0,
   'exciting_fermi': 2.0,
   'gallant_cerf': 

In [5]:
pprint(results[0])

{'cost': {'beautiful_bohr': 251156.24797256076,
          'competent_mayer': 406399.4018487107,
          'elated_haslett': 977691.1445767505,
          'elegant_gagarin': 627180.0496084697,
          'elegant_mendel': 348573.77294422314,
          'exciting_fermi': 493149.4427404745,
          'gallant_cerf': 647257.1157146845,
          'hopeful_sanderson': 823884.0054250445,
          'keen_visvesvaraya': 1194805.381073437,
          'laughing_mestorf': 840198.7682361763,
          'loving_engelbart': 1054403.388237939,
          'peaceful_austin': 868601.4376568777,
          'quizzical_elgamal': 870359.6940502283,
          'vigorous_mccarthy': 672948.548382829},
 'epsilon': {'beautiful_bohr': 2.0,
             'competent_mayer': 2.0,
             'elated_haslett': 2.0,
             'elegant_gagarin': 2.0,
             'elegant_mendel': 2.0,
             'exciting_fermi': 2.0,
             'gallant_cerf': 2.0,
             'hopeful_sanderson': 2.0,
             'keen_visvesvaraya'

In [ ]:
params = OptimizerParams(
    het_costs=het_costs,
    epsilons=epsilon,
    threads=14
)
options = SolverOptions(
    aggregate=True
)

opt = Optimizer(Platform, params)

summary_vanilla = opt.solve(options)

results.append({
    "instance_str": instance_str,
    "cost": het_costs,
    "epsilon": epsilon,
    "farmer_quantities": farmer_quantities,
    "summary_vanilla": summary_vanilla,
    "farmer_dirt_to_mill": farmer_dirt_to_mill,
    "famer_paved_to_mill": farmer_paved_to_mill,
})




============================= Optimizer Parameters =============================
---------------------------------- het_costs -----------------------------------
  {'beautiful_bohr': 251156.2479725607,
   'competent_mayer': 406399.4018487107,
   'elated_haslett': 977691.1445767505,
   'elegant_gagarin': 627180.0496084698,
   'elegant_mendel': 348573.77294422314,
   'exciting_fermi': 493149.4427404745,
   'gallant_cerf': 647257.1157146845,
   'hopeful_sanderson': 823884.0054250445,
   'keen_visvesvaraya': 1194805.3810734372,
   'laughing_mestorf': 840198.7682361763,
   'loving_engelbart': 1054403.388237939,
   'peaceful_austin': 868601.4376568777,
   'quizzical_elgamal': 870359.6940502283,
   'vigorous_mccarthy': 672948.548382829}
----------------------------------- epsilons -----------------------------------
  {'beautiful_bohr': 2.0,
   'competent_mayer': 2.0,
   'elated_haslett': 2.0,
   'elegant_gagarin': 2.0,
   'elegant_mendel': 2.0,
   'exciting_fermi': 2.0,
   'gallant_cerf': 

In [ ]:
pprint(results[0])

{'cost': {'beautiful_bohr': 251156.2479725607,
          'competent_mayer': 406399.4018487107,
          'elated_haslett': 977691.1445767505,
          'elegant_gagarin': 627180.0496084698,
          'elegant_mendel': 348573.77294422314,
          'exciting_fermi': 493149.4427404745,
          'gallant_cerf': 647257.1157146845,
          'hopeful_sanderson': 823884.0054250445,
          'keen_visvesvaraya': 1194805.3810734372,
          'laughing_mestorf': 840198.7682361763,
          'loving_engelbart': 1054403.388237939,
          'peaceful_austin': 868601.4376568777,
          'quizzical_elgamal': 870359.6940502283,
          'vigorous_mccarthy': 672948.548382829},
 'epsilon': {'beautiful_bohr': 2.0,
             'competent_mayer': 2.0,
             'elated_haslett': 2.0,
             'elegant_gagarin': 2.0,
             'elegant_mendel': 2.0,
             'exciting_fermi': 2.0,
             'gallant_cerf': 2.0,
             'hopeful_sanderson': 2.0,
             'keen_visvesvaraya'